In [ ]:
%%capture --no-stderr
# ! pip install -U langgraph langchain-community langchain-anthropic tavily-python pandas

In [ ]:
import os
import shutil
import sqlite3
import pandas as pd
import requests
import getpass
import re
import numpy as np
import openai
from langchain_core.tools import tool

In [ ]:
def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("ANTHROPIC_API_KEY")
_set_env("TAVILY_API_KEY")
_set_env("OPENAI_API_KEY")

# Recommended
_set_env("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Customer Support Bot Tutorial"

OPENAI_API_KEY: ··········
LANGCHAIN_API_KEY: ··········


In [ ]:
db_url = "https://storage.googleapis.com/benchmarks-artifacts/travel-db/travel2.sqlite"
local_file = "travel2.sqlite"
# The backup lets us restart for each tutorial section
backup_file = "travel2.backup.sqlite"
overwrite = False

In [ ]:
if overwrite or not os.path.exists(local_file):
    response = requests.get(db_url)
    response.raise_for_status()  # Ensure the request was successful
    with open(local_file, "wb") as f:
        f.write(response.content)
    # Backup - we will use this to "reset" our DB in each section
    shutil.copy(local_file, backup_file)

In [ ]:
# Convert the flights to present time for our tutorial
conn = sqlite3.connect(local_file)
cursor = conn.cursor()

In [ ]:
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';", conn
).name.tolist()

In [ ]:
tdf = {}
for t in tables:
    tdf[t] = pd.read_sql(f"SELECT * from {t}", conn)

In [ ]:
example_time = pd.to_datetime(
    tdf["flights"]["actual_departure"].replace("\\N", pd.NaT)
).max()
current_time = pd.to_datetime("now").tz_localize(example_time.tz)
time_diff = current_time - example_time

In [ ]:
tdf["bookings"]["book_date"] = (
    pd.to_datetime(tdf["bookings"]["book_date"].replace("\\N", pd.NaT), utc=True)
    + time_diff
)

In [ ]:
datetime_columns = [
    "scheduled_departure",
    "scheduled_arrival",
    "actual_departure",
    "actual_arrival",
]

In [ ]:
for column in datetime_columns:
    tdf["flights"][column] = (
        pd.to_datetime(tdf["flights"][column].replace("\\N", pd.NaT)) + time_diff
    )

In [ ]:
for table_name, df in tdf.items():
    df.to_sql(table_name, conn, if_exists="replace", index=False)
del df
del tdf
conn.commit()
conn.close()
db = local_file

In [ ]:
response = requests.get(
    "https://storage.googleapis.com/benchmarks-artifacts/travel-db/swiss_faq.md"
)
response.raise_for_status()
faq_text = response.text

docs = [{"page_content": txt} for txt in re.split(r"(?=\n##)", faq_text)]

In [ ]:
class VectorStoreRetriever:
    def __init__(self, docs: list, vectors: list, oai_client):
        self._arr = np.array(vectors)
        self._docs = docs
        self._client = oai_client

    @classmethod
    def from_docs(cls, docs, oai_client):
        embeddings = oai_client.embeddings.create(
            model="text-embedding-3-small", input=[doc["page_content"] for doc in docs]
        )
        vectors = [emb.embedding for emb in embeddings.data]
        return cls(docs, vectors, oai_client)

    def query(self, query: str, k: int = 5) -> list[dict]:
        embed = self._client.embeddings.create(
            model="text-embedding-3-small", input=[query]
        )
        # "@" is just a matrix multiplication in python
        scores = np.array(embed.data[0].embedding) @ self._arr.T
        top_k_idx = np.argpartition(scores, -k)[-k:]
        top_k_idx_sorted = top_k_idx[np.argsort(-scores[top_k_idx])]
        return [
            {**self._docs[idx], "similarity": scores[idx]} for idx in top_k_idx_sorted
        ]


retriever = VectorStoreRetriever.from_docs(docs, openai.Client())

In [ ]:
import logging
from telegram import Update
from telegram.ext import Updater, CommandHandler, MessageHandler, Filters, CallbackContext

In [ ]:
# Configuração do logging
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

# Dicionário simulando o inventário
inventario = {
    "caneta": 100,
    "lápis": 200,
    "borracha": 150,
    "caderno": 50,
    "pasta": 75
}

def start(update: Update, context: CallbackContext) -> None:
    """Envia uma mensagem quando o comando /start é emitido."""
    update.message.reply_text('Olá! Sou o bot do almoxarifado. Como posso ajudar?')

def help_command(update: Update, context: CallbackContext) -> None:
    """Envia uma mensagem quando o comando /help é emitido."""
    update.message.reply_text('Você pode me perguntar sobre a disponibilidade e quantidade de itens no estoque.')

def check_inventory(update: Update, context: CallbackContext) -> None:
    """Responde às consultas sobre o inventário."""
    query = update.message.text.lower()

    if "disponível" in query or "disponibilidade" in query:
        for item in inventario:
            if item in query:
                if inventario[item] > 0:
                    update.message.reply_text(f"Sim, {item} está disponível.")
                else:
                    update.message.reply_text(f"Desculpe, {item} não está disponível no momento.")
                return
        update.message.reply_text("Desculpe, não entendi qual item você está procurando.")

    elif "quantidade" in query or "quantos" in query:
        for item in inventario:
            if item in query:
                update.message.reply_text(f"Temos {inventario[item]} unidades de {item} em estoque.")
                return
        update.message.reply_text("Desculpe, não entendi qual item você está procurando.")

    else:
        update.message.reply_text("Desculpe, não entendi sua pergunta. Você pode perguntar sobre disponibilidade ou quantidade de itens.")

def main() -> None:
    """Inicia o bot."""
    # Substitua 'YOUR_TOKEN' pelo token do seu bot do Telegram
    updater = Updater("YOUR_TOKEN", use_context=True)

    dp = updater.dispatcher

    dp.add_handler(CommandHandler("start", start))
    dp.add_handler(CommandHandler("help", help_command))
    dp.add_handler(MessageHandler(Filters.text & ~Filters.command, check_inventory))

    updater.start_polling()
    updater.idle()

if __name__ == '__main__':
    main()

In [ ]:
!pip install --upgrade anthropic

In [ ]:
import anthropic
import os

In [ ]:
# Configurar a chave API
anthropic.api_key = "YOUR_ANTHROPIC_API_KEY"

In [ ]:
# Função para enviar mensagem e obter resposta
def get_response(prompt):
    try:
        response = anthropic.Completion.create(
            model="claude-2",  # ou outro modelo disponível
            prompt=f"Human: {prompt}\n\nAssistant:",
            max_tokens_to_sample=300,
            stop_sequences=["Human:"]
        )
        return response.completion.strip()
    except Exception as e:
        return f"Erro ao obter resposta: {str(e)}"

In [ ]:
# Loop principal do chatbot
print("Bem-vindo ao Chatbot Anthropic! Digite 'sair' para encerrar.")
while True:
    user_input = input("Você: ")
    if user_input.lower() == 'sair':
        print("Até logo!")
        break

    response = get_response(user_input)
    print("Chatbot:", response)

Bem-vindo ao Chatbot Anthropic! Digite 'sair' para encerrar.
Você: Boa noite
Chatbot: Erro ao obter resposta: module 'anthropic' has no attribute 'Completion'
Você: sair
Até logo!
